# FictiPay Churn Prediction Pipeline
This notebook contains the complete, production-grade end-to-end churn prediction pipeline for FictiPay customer accounts.

### Pipeline Structure:
1. **Phase 1: Big Data Handling** - Lazy ingestion & stratified sampling.
2. **Phase 2 & 3: Feature Engineering & Quality** - Extraction of 15+ behavioral features, winsorization, log transforms, and sparsity flags.
3. **Phase 4 & 5: Resampling & Model Training** - Logistic Regression baseline & LightGBM training with metrics.
4. **Phase 6: Hyperparameter Tuning** - Optuna Bayesian optimization.
5. **Phase 7: Explainability** - SHAP value summary and leakage audit.
6. **Phase 8: Business Recs** - Lift curves, risk deciles, and interventions.

## Step 1: Environment Setup
Import standard libraries and custom modules.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
import shap

# Import custom pipeline modules
from features import load_base_data, load_transactions_and_balances, extract_features, apply_feature_quality
from run_pipeline import evaluate_resampling_strategies, train_and_evaluate_models, run_hyperparameter_tuning, run_explainability, compute_lift_table

## Step 2: Load and Preprocess Data (10% Sample)
We use a reproducible 10% stratified sample for local modeling, feature quality checks, and tuning.

In [ ]:
train_labels, test_df, kyc_df, active_custs = load_base_data(sample_ratio=0.1, random_state=42)
trx_df, bal_df = load_transactions_and_balances(active_custs)

## Step 3: Feature Extraction and Preprocessing
Extract the 15 behavioral features and apply winsorization, log1p, and zero indicator flags to manage heavy skewness.

In [ ]:
features_df = extract_features(kyc_df, trx_df, bal_df, active_custs)

# Separate train features
train_cust_ids = train_labels["ACCOUNT_ID"].values
X_train_raw = features_df.loc[features_df.index.isin(train_cust_ids)].copy()
y_train_raw = train_labels.set_index("ACCOUNT_ID").loc[X_train_raw.index]["CHURN"]

# Apply Feature Quality adjustments
X_train_trans, train_stats = apply_feature_quality(X_train_raw, is_train=True)
X_train_trans.head(3)

## Step 4: Class Imbalance Resampling Strategy
Compare No Resampling, Class Weight, and Undersampling on a validation split (30% holdout).

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train_trans, y_train_raw, test_size=0.3, random_state=42, stratify=y_train_raw)

resampling_results = evaluate_resampling_strategies(X_tr, y_tr, X_val, y_val)
pd.DataFrame(resampling_results).T

## Step 5: Model Selection
Train Logistic Regression baseline and LightGBM model. Compare performance on AUC, AP, Prec@10%, and Recall@10%.

In [ ]:
model_evals, default_lgb = train_and_evaluate_models(X_tr, y_tr, X_val, y_val)
for name, metrics in model_evals.items():
    print(f"\n{name} Results:")
    print(f" - AUC-ROC:  {metrics['AUC']:.4f}")
    print(f" - Avg Prec:  {metrics['AP']:.4f}")
    print(f" - Prec@10%:  {metrics['P@10%']:.4f}")
    print(f" - Recall@10%:{metrics['R@10%']:.4f}")

## Step 6: Hyperparameter Tuning
Tune LightGBM classifier with Optuna (Bayesian optimization) over 25 trials.

In [ ]:
tuned_lgb, tuned_eval = run_hyperparameter_tuning(X_tr, y_tr, X_val, y_val)
print(f"\nTuned LGBM Validation AUC: {tuned_eval['AUC']:.4f}")

## Step 7: Model Explainability
Produce SHAP Beeswarm plot and dependency plots to explain model decision criteria.

In [ ]:
top_feats = run_explainability(tuned_lgb, X_val)
print("Top SHAP features:", top_feats)

## Step 8: Business Recommendations & Lift Curve
Generate the risk decile lift table to frame the outreach rule.

In [ ]:
lift_table = compute_lift_table(y_val, tuned_eval["preds"])
lift_table